# Automated QA System: Low-Latency Fruit Freshness Detection

Klasifikasi citra untuk mengotomatisasi inspeksi mutu buah di meja pemaketan (packing station) quick-commerce. Tujuannya memblokir produk cacat sebelum terkirim, sehingga biaya refund logistik dan customer churn berkurang.

Metrik utama bisnis: recall untuk buah busuk (Rotten) dan latensi inferensi P95 di bawah 100 ms.

# Import Library dan Konfigurasi

In [ ]:
import os
import json
import time
import shutil
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, applications, callbacks, optimizers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

USE_DRIVE = True
CKPT_DIR = "checkpoints"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CKPT_DIR = "/content/drive/MyDrive/fruit_qa"
    except ImportError:
        pass
os.makedirs(CKPT_DIR, exist_ok=True)

KAGGLE_PATH = kagglehub.dataset_download("ulnnproject/food-freshness-dataset")

@dataclass
class PackingStationConfig:
    dataset_root: str = str(Path(KAGGLE_PATH) / "Dataset")
    cache_df_path: str = "dataset_metadata.parquet"
    checkpoint_path: str = os.path.join(CKPT_DIR, "fruit_qa_best.keras")
    checkpoint_ft_path: str = os.path.join(CKPT_DIR, "fruit_qa_best_ft.keras")
    final_path: str = os.path.join(CKPT_DIR, "fruit_qa_final.keras")
    export_dir: str = os.path.join(CKPT_DIR, "exports")
    eval_dir: str = os.path.join(CKPT_DIR, "eval")
    img_size: tuple = (224, 224)
    channels: int = 3
    val_split: float = 0.15
    test_split: float = 0.15
    batch_size: int = 32
    seed: int = 42

    epochs_phase1: int = 8
    lr_phase1: float = 1e-3
    epochs_phase2: int = 12
    lr_phase2: float = 2e-5
    unfreeze_last_n: int = 60

    dropout_rate_1: float = 0.4
    dropout_rate_2: float = 0.3
    dense_units: int = 256
    l2_reg: float = 1e-4
    rotten_boost: float = 1.5

    target_recall: float = 0.95
    latency_threshold_ms: int = 100
    latency_warmup_runs: int = 10
    latency_benchmark_runs: int = 100

config = PackingStationConfig()
tf.keras.utils.set_random_seed(config.seed)
os.makedirs(config.export_dir, exist_ok=True)
os.makedirs(config.eval_dir, exist_ok=True)

print("Checkpoint   :", CKPT_DIR)
print("Dataset root :", config.dataset_root)
print("GPU          :", [g.name for g in tf.config.list_physical_devices("GPU")] or "tidak terdeteksi")

---
## 1. EDA

Folder dataset dipindai sekali lalu hasilnya disimpan sebagai metadata. Dua nama folder yang salah ketik (`FreshCapciscum`, `FreshOkara`) disamakan dengan pasangan Rotten-nya supaya tidak terhitung sebagai kelas berbeda.

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_FIX = {"FreshCapciscum": "FreshCapsicum", "FreshOkara": "FreshOkra"}

def scan_dataset(root):
    rows = []
    for category in ("Fresh", "Rotten"):
        for sub in sorted((Path(root) / category).iterdir()):
            if not sub.is_dir():
                continue
            name = LABEL_FIX.get(sub.name, sub.name)
            for f in sub.iterdir():
                if f.suffix.lower() in IMAGE_EXT:
                    rows.append((str(f), name, category))
    return pd.DataFrame(rows, columns=["path", "label_name", "category"])

cache_valid = False
if os.path.exists(config.cache_df_path):
    df = pd.read_parquet(config.cache_df_path)
    cache_valid = len(df) > 0 and os.path.exists(df["path"].iloc[0])

if not cache_valid:
    df = scan_dataset(config.dataset_root)
    df.to_parquet(config.cache_df_path)

df["label_name"] = df["label_name"].replace(LABEL_FIX)

df_dist = (df.groupby(["label_name", "category"]).size()
             .reset_index(name="count")
             .sort_values("count", ascending=False)
             .reset_index(drop=True))
imbalance_ratio = df_dist["count"].max() / df_dist["count"].min()

print(f"Total gambar    : {len(df):,}")
print(f"Total kelas     : {len(df_dist)}")
print(f"Imbalance ratio : {imbalance_ratio:.1f}x")
print()
print(df_dist.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
colors = ["#e74c3c" if c == "Rotten" else "#2ecc71" for c in df_dist["category"]]
axes[0].barh(df_dist["label_name"], df_dist["count"], color=colors)
axes[0].invert_yaxis()
axes[0].axvline(df_dist["count"].mean(), color="navy", linestyle="--", alpha=0.7)
axes[0].set_title("Jumlah gambar per kelas")
axes[0].set_xlabel("Jumlah gambar")
axes[0].legend(handles=[
    Patch(facecolor="#2ecc71", label="Fresh"),
    Patch(facecolor="#e74c3c", label="Rotten"),
    plt.Line2D([0], [0], color="navy", linestyle="--", label=f"Rata-rata ({df_dist['count'].mean():.0f})"),
], fontsize=8)

cat_counts = df["category"].value_counts()
axes[1].pie(cat_counts[["Fresh", "Rotten"]], labels=["Fresh", "Rotten"], autopct="%1.1f%%",
            colors=["#2ecc71", "#e74c3c"], startangle=90)
axes[1].set_title("Fresh vs Rotten")
plt.tight_layout()
plt.show()

In [ ]:
class_list = sorted(df["label_name"].unique())
n_samples = 4
classes_per_row = 2
n_rows = int(np.ceil(len(class_list) / classes_per_row))

fig, axes = plt.subplots(n_rows, classes_per_row * n_samples,
                         figsize=(classes_per_row * n_samples * 2.2, n_rows * 2.3))
axes = axes.reshape(n_rows, -1)

for k, name in enumerate(class_list):
    r, c0 = divmod(k, classes_per_row)
    c0 *= n_samples
    paths = df.loc[df["label_name"] == name, "path"].sample(n_samples, random_state=config.seed)
    for j, p in enumerate(paths):
        ax = axes[r, c0 + j]
        ax.imshow(Image.open(p).convert("RGB").resize((160, 160)))
        ax.axis("off")
        if j == 0:
            color = "#e74c3c" if name.startswith("Rotten") else "#1e8449"
            ax.set_title(name, fontsize=9, loc="left", color=color, fontweight="bold")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
check = df.sample(500, random_state=config.seed)
widths, heights, brightness, unreadable = [], [], [], 0
for p in check["path"]:
    try:
        with Image.open(p) as im:
            widths.append(im.width)
            heights.append(im.height)
            brightness.append(np.asarray(im.convert("L")).mean())
    except Exception:
        unreadable += 1

summary = pd.DataFrame({
    "lebar": pd.Series(widths).describe()[["min", "50%", "max"]],
    "tinggi": pd.Series(heights).describe()[["min", "50%", "max"]],
    "brightness": pd.Series(brightness).describe()[["min", "50%", "max"]],
}).round(1)
print(f"Sampel diperiksa : {len(check)}")
print(f"Tidak terbaca    : {unreadable}")
print(summary)

---
## 2. Data Pipeline dan Augmentasi

Split stratified 70/15/15 per kelas: train untuk melatih, validasi untuk early stopping dan pemilihan threshold, test hanya dipakai sekali di evaluasi akhir. Bobot kelas dihitung dari frekuensi dengan akar kuadrat supaya kelas kecil tidak mendapat bobot ekstrem, lalu kelas Rotten dinaikkan sedikit.

In [ ]:
class_names = sorted(df["label_name"].unique())
num_classes = len(class_names)
label_to_idx = {name: i for i, name in enumerate(class_names)}
df["label"] = df["label_name"].map(label_to_idx)

is_rotten_class = np.array([n.startswith("Rotten") for n in class_names])
rotten_idx = np.where(is_rotten_class)[0]

df_trainval, df_test = train_test_split(
    df, test_size=config.test_split, random_state=config.seed, stratify=df["label"]
)
df_train, df_val = train_test_split(
    df_trainval, test_size=config.val_split / (1 - config.test_split),
    random_state=config.seed, stratify=df_trainval["label"]
)

train_counts = df_train["label"].value_counts().sort_index()
raw_weights = len(df_train) / (num_classes * train_counts)
class_weights = {int(i): float(w ** 0.5) for i, w in raw_weights.items()}
for i in rotten_idx:
    class_weights[int(i)] *= config.rotten_boost

weight_table = pd.DataFrame({
    "kelas": class_names,
    "jumlah_train": [int(train_counts[i]) for i in range(num_classes)],
    "bobot": [round(class_weights[i], 3) for i in range(num_classes)],
})
print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(weight_table.to_string(index=False))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, config.img_size)
    img.set_shape([*config.img_size, 3])
    return img, tf.one_hot(label, num_classes)

def make_dataset(frame, training):
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["label"].values))
    if training:
        ds = ds.shuffle(len(frame), seed=config.seed)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE, deterministic=not training)
    ds = ds.ignore_errors()
    return ds.batch(config.batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(df_train, training=True)
val_ds = make_dataset(df_val, training=False)
test_ds = make_dataset(df_test, training=False)

def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.10),
        layers.RandomZoom(height_factor=(-0.1, 0.1)),
    ], name="packing_augmentation")

augmentation = build_augmentation()
sample_images, _ = next(iter(train_ds.take(1)))
sample_img = sample_images[0:1]

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes[0, 0].imshow(sample_img[0].numpy().astype(np.uint8))
axes[0, 0].set_title("Original", fontweight="bold")
for i, ax in enumerate(axes.flat[1:], start=1):
    aug = augmentation(sample_img, training=True)[0].numpy()
    ax.imshow(np.clip(aug, 0, 255).astype(np.uint8))
    ax.set_title(f"Augmentasi {i}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 3. Arsitektur Model: EfficientNetB0 Transfer Learning

Phase 1 melatih classification head dengan backbone dibekukan. Phase 2 membuka sebagian lapisan atas backbone dengan learning rate kecil. Backbone dibangun sebagai model terpisah yang dipanggil dengan `training=False`, sehingga lapisan BatchNormalization tetap memakai statistik ImageNet selama fine-tuning.

In [ ]:
def build_model(num_classes, config):
    inputs = layers.Input(shape=(*config.img_size, config.channels), name="camera_feed")
    x = build_augmentation()(inputs)

    backbone = applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(*config.img_size, config.channels)
    )
    backbone.trainable = False
    x = backbone(x, training=False)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dropout(config.dropout_rate_1, name="drop_1")(x)
    x = layers.Dense(config.dense_units, activation="relu",
                     kernel_regularizer=regularizers.l2(config.l2_reg), name="dense_1")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dropout(config.dropout_rate_2, name="drop_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="qa_decision")(x)

    model = tf.keras.Model(inputs, outputs, name="FruitQA_EfficientNetB0")
    return model, backbone

def compile_model(model, lr):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

def count_trainable(model):
    return sum(int(np.prod(v.shape)) for v in model.trainable_variables)

qa_model, backbone = build_model(num_classes, config)
compile_model(qa_model, config.lr_phase1)
qa_model.summary(line_length=100)
print(f"\nTrainable params: {count_trainable(qa_model):,}")

---
## 4. Training Phase 1: Feature Extraction

In [ ]:
callbacks_phase1 = [
    callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=3,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(filepath=config.checkpoint_path, monitor="val_accuracy",
                              mode="max", save_best_only=True, verbose=1),
]

history_p1 = qa_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.epochs_phase1,
    callbacks=callbacks_phase1,
    class_weight=class_weights,
    verbose=1,
)

In [ ]:
def plot_learning_curves(history, title):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        epochs = range(1, len(h[metric]) + 1)
        ax.plot(epochs, h[metric], "o-", ms=3, label="Train")
        ax.plot(epochs, h[f"val_{metric}"], "o-", ms=3, label="Val")
        val = np.array(h[f"val_{metric}"])
        best = int(val.argmin() if metric == "loss" else val.argmax())
        ax.axvline(best + 1, color="gray", linestyle="--", alpha=0.6, label=f"Terbaik (epoch {best + 1})")
        ax.set_title(metric)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(title, fontweight="bold")
    plt.tight_layout()
    plt.show()

def summarize_history(history):
    h = history.history
    best = int(np.argmax(h["val_accuracy"]))
    return {
        "epoch_dijalankan": len(h["loss"]),
        "epoch_terbaik": best + 1,
        "val_accuracy": h["val_accuracy"][best],
        "val_loss": h["val_loss"][best],
        "train_accuracy": h["accuracy"][best],
        "train_loss": h["loss"][best],
    }

plot_learning_curves(history_p1, "Phase 1 (Feature Extraction)")
summary_p1 = summarize_history(history_p1)
print(pd.Series(summary_p1).round(4).to_string())

---
## 5. Training Phase 2: Fine-Tuning

In [ ]:
backbone.trainable = True
for layer in backbone.layers[:-config.unfreeze_last_n]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model(qa_model, config.lr_phase2)

n_trainable_layers = sum(l.trainable for l in backbone.layers)
print(f"Lapisan backbone       : {len(backbone.layers)}")
print(f"Lapisan dilatih        : {n_trainable_layers} (BatchNorm tetap beku)")
print(f"Trainable params       : {count_trainable(qa_model):,}")
print(f"Learning rate          : {config.lr_phase2}")

In [ ]:
callbacks_phase2 = [
    callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint(filepath=config.checkpoint_ft_path, monitor="val_accuracy",
                              mode="max", save_best_only=True, verbose=1),
]

history_p2 = qa_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.epochs_phase2,
    callbacks=callbacks_phase2,
    class_weight=class_weights,
    verbose=1,
)

In [ ]:
plot_learning_curves(history_p2, "Phase 2 (Fine-Tuning)")
summary_p2 = summarize_history(history_p2)

comparison = pd.DataFrame({"Phase 1": summary_p1, "Phase 2": summary_p2}).T
comparison["delta_val_accuracy"] = comparison["val_accuracy"] - comparison.loc["Phase 1", "val_accuracy"]
print(comparison[["epoch_terbaik", "val_accuracy", "val_loss", "train_accuracy", "delta_val_accuracy"]].round(4))

best_source = (config.checkpoint_ft_path
               if summary_p2["val_accuracy"] >= summary_p1["val_accuracy"]
               else config.checkpoint_path)
shutil.copy(best_source, config.final_path)
print(f"\nCheckpoint akhir: {os.path.basename(best_source)} disalin ke {config.final_path}")

---
## 6. Evaluasi

Model dimuat dari file checkpoint di disk, bukan dari variabel di memori, sehingga yang dievaluasi sama persis dengan yang diekspor. Jika file tidak ada, jumlah kelasnya tidak cocok, atau akurasinya mendekati tebakan acak, eksekusi dihentikan. Evaluasi akhir memakai test set; validasi hanya dipakai untuk memilih threshold. Selain evaluasi per kelas, evaluasi biner Fresh vs Rotten dihitung dari jumlah probabilitas seluruh kelas Rotten, karena yang menentukan bisnis adalah apakah buah busuk berhasil diblokir, bukan apakah jenis buahnya tertebak benar.

In [ ]:
candidates = [config.final_path, config.checkpoint_ft_path, config.checkpoint_path]
model_path = next((p for p in candidates if os.path.exists(p)), None)
if model_path is None:
    raise FileNotFoundError(
        "Tidak ada checkpoint terlatih. Jalankan section 4 dan 5, atau salin file .keras ke " + CKPT_DIR)

final_model = tf.keras.models.load_model(model_path)
n_outputs = final_model.output_shape[-1]
if n_outputs != num_classes:
    raise ValueError(
        f"{os.path.basename(model_path)} punya {n_outputs} output, sedangkan data saat ini {num_classes} kelas. "
        "Checkpoint ini berasal dari versi notebook yang lain, latih ulang.")

print("Model dimuat dari :", model_path)
print("Terakhir diubah   :", time.ctime(os.path.getmtime(model_path)))

predict_fn = tf.function(lambda x: final_model(x, training=False))

def predict_dataset(ds):
    labels_all, probs_all = [], []
    for images, labels in ds:
        probs_all.append(predict_fn(images).numpy())
        labels_all.append(labels.numpy().argmax(axis=1))
    return np.concatenate(labels_all), np.concatenate(probs_all)

y_val, prob_val = predict_dataset(val_ds)
y_true, y_prob = predict_dataset(test_ds)
y_pred = y_prob.argmax(axis=1)

val_accuracy = (y_val == prob_val.argmax(axis=1)).mean()
test_accuracy = (y_true == y_pred).mean()
print(f"Sampel validasi   : {len(y_val):,} | accuracy {val_accuracy:.4f}")
print(f"Sampel test       : {len(y_true):,} | accuracy {test_accuracy:.4f}")

if test_accuracy < 0.30:
    raise RuntimeError("Akurasi test mendekati tebakan acak. Model tampaknya belum terlatih, evaluasi dihentikan.")

In [ ]:
label_ids = np.arange(num_classes)
report = classification_report(y_true, y_pred, labels=label_ids, target_names=class_names,
                               output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, labels=label_ids, target_names=class_names,
                            digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=label_ids)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

true_rotten = is_rotten_class[y_true]
pred_rotten = is_rotten_class[y_pred]
cm_bin = confusion_matrix(true_rotten, pred_rotten, labels=[False, True])

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [3, 1.2]})
ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(
    ax=axes[0], cmap="Blues", xticks_rotation=90, include_values=False, colorbar=False)
axes[0].set_title("26 kelas (dinormalisasi per baris)")
ConfusionMatrixDisplay(cm_bin, display_labels=["Fresh", "Rotten"]).plot(
    ax=axes[1], cmap="Blues", values_format="d", colorbar=False)
axes[1].set_title("Fresh vs Rotten (argmax)")
plt.tight_layout()
fig.savefig(os.path.join(config.eval_dir, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

rows = []
for i in rotten_idx:
    total = cm[i].sum()
    escaped = cm[i][~is_rotten_class].sum()
    rows.append({"kelas": class_names[i], "sampel": total, "lolos_sebagai_fresh": escaped,
                 "miss_rate_%": round(100 * escaped / total, 2)})
print("Buah busuk yang lolos sebagai kelas Fresh (argmax):")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def threshold_metrics(p, truth, t):
    pred = p >= t
    tp = (pred & truth).sum()
    fn = (~pred & truth).sum()
    fp = (pred & ~truth).sum()
    return {"threshold": float(t), "recall": tp / (tp + fn), "precision": tp / max(tp + fp, 1),
            "false_block_rate": fp / (~truth).sum()}

p_rotten_val = prob_val[:, rotten_idx].sum(axis=1)
true_rotten_val = is_rotten_class[y_val]
p_rotten = y_prob[:, rotten_idx].sum(axis=1)

thresholds = np.linspace(0.02, 0.98, 97)
sweep = pd.DataFrame([threshold_metrics(p_rotten_val, true_rotten_val, t) for t in thresholds])

eligible = sweep[sweep["recall"] >= config.target_recall]
operating_threshold = float(eligible["threshold"].max()) if len(eligible) else float(sweep["threshold"].min())
val_point = threshold_metrics(p_rotten_val, true_rotten_val, operating_threshold)
operating_point = threshold_metrics(p_rotten, true_rotten, operating_threshold)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sweep["threshold"], sweep["recall"], label="Recall Rotten")
ax.plot(sweep["threshold"], sweep["precision"], label="Precision Rotten")
ax.plot(sweep["threshold"], sweep["false_block_rate"], label="Fresh terblokir")
ax.axhline(config.target_recall, color="gray", linestyle=":")
ax.axvline(operating_threshold, color="red", linestyle="--", label=f"Threshold dipilih ({operating_threshold:.2f})")
ax.set_xlabel("Ambang P(Rotten), kurva dari data validasi")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(config.eval_dir, "threshold_sweep.png"), dpi=150, bbox_inches="tight")
plt.show()

argmax_recall = cm_bin[1, 1] / cm_bin[1].sum()
argmax_precision = cm_bin[1, 1] / cm_bin[:, 1].sum()
print(f"Test, argmax            : recall Rotten {argmax_recall:.4f} | precision {argmax_precision:.4f}")
print(f"Validasi, threshold {operating_threshold:.2f} : recall Rotten {val_point['recall']:.4f} | "
      f"precision {val_point['precision']:.4f}")
print(f"Test, threshold {operating_threshold:.2f}     : recall Rotten {operating_point['recall']:.4f} | "
      f"precision {operating_point['precision']:.4f} | Fresh terblokir {operating_point['false_block_rate']:.4f}")

In [ ]:
report_df = pd.DataFrame(report).T.loc[class_names]
x = np.arange(num_classes)
w = 0.27

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(x - w, report_df["recall"], w, label="Recall", color="#e74c3c")
ax.bar(x, report_df["precision"], w, label="Precision", color="steelblue")
ax.bar(x + w, report_df["f1-score"], w, label="F1", color="darkorchid")
ax.axhline(config.target_recall, color="gray", linestyle="--", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Metrik per kelas (test set)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(config.eval_dir, "per_class_metrics.png"), dpi=150, bbox_inches="tight")
plt.show()

weak = report_df[report_df["recall"] < 0.90][["precision", "recall", "f1-score", "support"]]
print("Kelas dengan recall di bawah 0.90:")
print(weak.round(3).to_string() if len(weak) else "tidak ada")

In [ ]:
confidence = y_prob.max(axis=1)
rows = []
for t in (0.0, 0.5, 0.7, 0.9):
    keep = confidence >= t
    rows.append({
        "min_confidence": t,
        "diproses_otomatis_%": round(100 * keep.mean(), 1),
        "accuracy_otomatis": round((y_true[keep] == y_pred[keep]).mean(), 4),
        "dikirim_ke_review_manual": int((~keep).sum()),
    })
print(pd.DataFrame(rows).to_string(index=False))

---
## 7. SLA Test: Latency Profiling

Model dianggap layak deploy jika latensi P95 per gambar di bawah 100 ms. Pengukuran memakai satu gambar per panggilan dan `tf.function`, bukan `model.predict`, karena `predict` menambah overhead yang tidak ada di sistem produksi.

In [ ]:
dummy_frame = tf.constant(np.random.rand(1, *config.img_size, 3).astype(np.float32) * 255)

for _ in range(config.latency_warmup_runs):
    predict_fn(dummy_frame).numpy()

latencies = []
for _ in range(config.latency_benchmark_runs):
    t0 = time.perf_counter()
    predict_fn(dummy_frame).numpy()
    latencies.append((time.perf_counter() - t0) * 1000)
latencies = np.array(latencies)

p50, p95, p99 = np.percentile(latencies, [50, 95, 99])
mean_lat = latencies.mean()
violations = int((latencies > config.latency_threshold_ms).sum())

print(f"Mean : {mean_lat:.2f} ms (std {latencies.std():.2f})")
print(f"P50  : {p50:.2f} ms")
print(f"P95  : {p95:.2f} ms (batas {config.latency_threshold_ms} ms)")
print(f"P99  : {p99:.2f} ms")
print(f"Pelanggaran SLA : {violations}/{len(latencies)}")
print(f"Throughput      : {1000 / mean_lat:.1f} gambar/detik")
print("Status          :", "memenuhi SLA" if p95 <= config.latency_threshold_ms else "tidak memenuhi SLA")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(latencies, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(p95, color="red", linestyle="--", label=f"P95 {p95:.1f} ms")
axes[0].axvline(config.latency_threshold_ms, color="orange", label=f"SLA {config.latency_threshold_ms} ms")
axes[0].set_xlabel("ms")
axes[0].set_title("Distribusi latensi")
axes[0].legend()
axes[1].plot(latencies, color="steelblue", alpha=0.7, lw=0.8)
axes[1].axhline(config.latency_threshold_ms, color="orange")
axes[1].set_xlabel("Inferensi ke-")
axes[1].set_ylabel("ms")
axes[1].set_title("Urutan waktu")
plt.tight_layout()
plt.show()

---
## 8. Business Scorecard dan Export

In [ ]:
rotten_recalls = [report[class_names[i]]["recall"] for i in rotten_idx]

scorecard = pd.Series({
    "Accuracy (26 kelas)": report["accuracy"],
    "Macro precision": report["macro avg"]["precision"],
    "Macro recall": report["macro avg"]["recall"],
    "Macro F1": report["macro avg"]["f1-score"],
    "Recall Rotten, argmax (biner)": argmax_recall,
    f"Recall Rotten, threshold {operating_threshold:.2f}": operating_point["recall"],
    f"Precision Rotten, threshold {operating_threshold:.2f}": operating_point["precision"],
    "Fresh terblokir salah": operating_point["false_block_rate"],
    "Rata-rata recall per kelas Rotten": float(np.mean(rotten_recalls)),
})
print("Hasil pada test set")
print(scorecard.round(4).to_string())
print(f"\nP95 latency : {p95:.1f} ms (batas {config.latency_threshold_ms} ms)")

deploy_ready = (operating_point["recall"] >= config.target_recall) and (p95 <= config.latency_threshold_ms)
print("Status akhir:", "layak deploy" if deploy_ready else "belum layak deploy")

In [ ]:
pd.DataFrame(report).T.to_csv(os.path.join(config.eval_dir, "classification_report.csv"))
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(os.path.join(config.eval_dir, "confusion_matrix_26_kelas.csv"))
pd.DataFrame(cm_bin, index=["true_Fresh", "true_Rotten"], columns=["pred_Fresh", "pred_Rotten"]).to_csv(
    os.path.join(config.eval_dir, "confusion_matrix_fresh_vs_rotten.csv"))
sweep.to_csv(os.path.join(config.eval_dir, "threshold_sweep.csv"), index=False)

results = {
    "model_file": os.path.basename(model_path),
    "model_modified": time.ctime(os.path.getmtime(model_path)),
    "n_train": len(df_train), "n_val": len(df_val), "n_test": len(df_test),
    "num_classes": num_classes,
    "val_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(report["macro avg"]["f1-score"]),
    "test_rotten_recall_argmax": float(argmax_recall),
    "test_rotten_precision_argmax": float(argmax_precision),
    "operating_threshold_from_val": operating_threshold,
    "test_rotten_recall_at_threshold": float(operating_point["recall"]),
    "test_rotten_precision_at_threshold": float(operating_point["precision"]),
    "test_fresh_blocked_at_threshold": float(operating_point["false_block_rate"]),
    "latency_p95_ms": float(p95),
}
with open(os.path.join(config.eval_dir, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

archive = shutil.make_archive("evaluasi_fruit_qa", "zip", config.eval_dir)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Arsip:", archive)

In [ ]:
saved_model_dir = os.path.join(config.export_dir, "fruit_qa_v1_savedmodel")
keras_path = os.path.join(config.export_dir, "fruit_qa_v1.keras")
tflite_path = os.path.join(config.export_dir, "fruit_qa_v1.tflite")

inference_input = layers.Input(shape=(*config.img_size, config.channels), name="camera_feed")
x = inference_input
for layer in final_model.layers:
    if isinstance(layer, layers.InputLayer) or layer.name == "packing_augmentation":
        continue
    x = layer(x)
inference_model = tf.keras.Model(inference_input, x, name="FruitQA_inference")

inference_model.save(keras_path)
inference_model.export(saved_model_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

saved_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fn in os.walk(saved_model_dir) for f in fn) / 1e6
tflite_mb = os.path.getsize(tflite_path) / 1e6
print(f"SavedModel : {saved_mb:.1f} MB")
print(f"TFLite     : {tflite_mb:.1f} MB")

interpreter = tf.lite.Interpreter(model_content=tflite_model, num_threads=4)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

def tflite_predict(img):
    interpreter.set_tensor(inp["index"], img[None].astype(np.float32))
    interpreter.invoke()
    return interpreter.get_tensor(out["index"])[0]

check_images, _ = next(iter(val_ds.take(1)))
check_images = check_images.numpy()
keras_top = inference_model.predict(check_images, verbose=0).argmax(axis=1)
tflite_top = np.array([tflite_predict(im).argmax() for im in check_images])
print(f"Kesesuaian prediksi Keras vs TFLite: {(keras_top == tflite_top).mean():.3f} ({len(check_images)} gambar)")

for _ in range(5):
    tflite_predict(check_images[0])
tflite_ms = []
for _ in range(config.latency_benchmark_runs):
    t0 = time.perf_counter()
    tflite_predict(check_images[0])
    tflite_ms.append((time.perf_counter() - t0) * 1000)
print(f"Latensi TFLite (CPU, 4 thread): P50 {np.percentile(tflite_ms, 50):.1f} ms | P95 {np.percentile(tflite_ms, 95):.1f} ms")